
# การรัน Deterministic Inference

workflow พื้นฐานสำหรับ deterministic inference

ตัวอย่างนี้จะแสดงวิธีรัน inference workflow แบบง่าย เพื่อสร้าง deterministic forecast เบื้องต้น โดยใช้หนึ่งในโมเดลที่มีมาให้ภายใน Earth-2 Inference Studio

ในตัวอย่างนี้คุณจะได้เรียนรู้:

- วิธีสร้างอินสแตนซ์ของโมเดลพยากรณ์ที่มีมาให้ในระบบ
- วิธีสร้างแหล่งข้อมูลและออบเจ็กต์ IO
- วิธีรัน workflow พื้นฐานที่มีมาให้ในระบบ
- วิธีทำ post-processing กับผลลัพธ์


In [ ]:
# /// script
# dependencies = [
#   "earth2studio[dlwp] @ git+https://github.com/NVIDIA/earth2studio.git",
#   "cartopy",
# ]
# ///

## การเตรียมองค์ประกอบ
workflow ทุกตัวภายใน Earth2Studio จำเป็นต้องได้รับคอมโพเนนต์ที่สร้างไว้ล่วงหน้าแล้วส่งเข้าไปให้ใช้งาน ในตัวอย่างนี้เราจะดูเวิร์กโฟลว์พื้นฐานที่สุดคือ
:py:meth:`earth2studio.run.deterministic`.



.. literalinclude:: ../../earth2studio/run.py
   :language: python
   :start-after: # sphinx - deterministic start
   :end-before: # sphinx - deterministic end



ดังนั้น เราจึงต้องมีองค์ประกอบต่อไปนี้:

- Prognostic Model: ใช้ FourCastNet รุ่น :py:class:`earth2studio.models.px.FCN` ในตัว
- Datasource: ดึงข้อมูลจาก GFS data API ผ่าน :py:class:`earth2studio.data.GFS`.
- IO Backend: บันทึกผลลัพธ์ลงใน Zarr store ผ่าน :py:class:`earth2studio.io.ZarrBackend`.



In [ ]:
import os

os.makedirs("outputs", exist_ok=True)
from dotenv import load_dotenv

load_dotenv()  # สิ่งที่ต้องทำ: สร้างฟังก์ชันการเตรียมตัวอย่างทั่วไป

from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.models.px import DLWP

# โหลดmodel packageเริ่มต้นซึ่งดาวน์โหลดcheckpointจาก NGC
package = DLWP.load_default_package()
model = DLWP.load_model(package)

# สร้างแหล่งข้อมูล
data = GFS()

# สร้างตัวจัดการ IO เก็บไว้ในหน่วยความจำ
io = ZarrBackend()

## การรัน Workflow
เมื่อคอมโพเนนต์ทั้งหมดเริ่มต้นแล้ว การรัน workflow จะเป็นโค้ด Python บรรทัดเดียว
เวิร์กโฟลว์จะส่งคืนอ็อบเจ็กต์ IO ที่ระบุกลับไปยังผู้ใช้ ซึ่งสามารถนำมาใช้ได้
จากนั้นจึงโพสต์กระบวนการ บางตัวมี API เพิ่มเติมซึ่งมีประโยชน์สำหรับ post-processing หรือ
บันทึกเป็นไฟล์ ตรวจสอบเอกสาร API สำหรับข้อมูลเพิ่มเติม

สำหรับการพยากรณ์ เราจะคาดการณ์เป็นเวลาสองวัน (ซึ่งจะถูกดำเนินการเป็น batch) สำหรับ
20 forecast steps ซึ่งก็คือ 5 วัน



In [ ]:
import earth2studio.run as run

nsteps = 20
io = run.deterministic(["2026-04-23"], nsteps, model, data, io)

print(io.root.tree())

## การทำ Post-Processing
ขั้นตอนสุดท้ายคือการนำผลลัพธ์มาทำ post-process ต่อ Cartopy เป็นไลบรารีที่ยอดเยี่ยมสำหรับการพล็อต
ฟิลด์ข้อมูลบน projection ของทรงกลม ตรงนี้เราจะพลอตอุณหภูมิที่ 2 เมตร
(t2m) 1 วันใน forecast

สังเกตว่าฟังก์ชัน Zarr IO มี API เพิ่มเติมสำหรับใช้เข้าถึงและจัดการข้อมูลที่เก็บไว้



In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

forecast = "2026-04-23"
variable = "t2m"
step = 4  # lead time = 24 ชม

plt.close("all")
# สร้างโรบินสัน projection
projection = ccrs.Robinson()

# สร้างรูปและแกนด้วย projection ที่ระบุ
fig, ax = plt.subplots(subplot_kw={"projection": projection}, figsize=(10, 6))

# พล็อตฟิลด์โดยใช้ pcolormesh
im = ax.pcolormesh(
    io["lon"][:],
    io["lat"][:],
    io[variable][0, step],
    transform=ccrs.PlateCarree(),
    cmap="Spectral_r",
)

# ตั้งชื่อเรื่อง
ax.set_title(f"{forecast} - Lead time: {6*step}hrs")

# เพิ่มแนวชายฝั่งและเส้นกริด
ax.coastlines()
ax.gridlines()
plt.savefig("outputs/01_t2m_prediction.jpg")